# 03 — CNN training

**Goal**: the target architecture. A 4-block CNN with BatchNorm, dropout, AdamW, and cosine LR scheduling. Same data, same loss, same Lightning trainer as the baseline — only the architecture changes.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.training import run_experiment
from src.utils import load_config
from src.models import CNNModel

## 1. Architecture summary

Before training, let's see what we're working with.

In [ ]:
model = CNNModel(num_classes=2)
print(model)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTrainable parameters: {n_params:,}')

## 2. Load the CNN config

In [ ]:
cfg = load_config('../configs/cnn.yaml')

import json
print(json.dumps(cfg, indent=2))

## 3. Train

On CPU this takes a few minutes per epoch at 64×64; on GPU under a minute. Early stopping (patience 4) will likely kick in before the 15-epoch cap.

In [ ]:
result = run_experiment(cfg)

## 4. Final metrics

In [ ]:
for k, v in result['metrics'].items():
    print(f'{k:15s}: {v:.4f}')
print(f"\nBest checkpoint: {result['best_ckpt']}")

## 5. Quick visualisation of the learning history

The full comparison plot lives in [notebook 04](04_model_comparison.ipynb). Here we just sanity-check the CNN alone.

In [ ]:
import matplotlib.pyplot as plt

history = result['history']
trim = lambda v: [x for x in v if x != 0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(trim(history['train_loss']), label='train', color='#f97316')
axes[0].plot(trim(history['val_loss']),   label='val',   color='#f97316', ls='--', alpha=0.7)
axes[0].set_title('Cross-entropy loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(trim(history['train_acc']), label='train', color='#f97316')
axes[1].plot(trim(history['val_acc']),   label='val',   color='#f97316', ls='--', alpha=0.7)
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout(); plt.show()

Next: [notebook 04](04_model_comparison.ipynb) — the head-to-head analysis with both models loaded from checkpoints.